In [1]:
!pip -q install transformers datasets accelerate scikit-learn sentencepiece

In [2]:
# ─────────────────────────────────────────────
# CELL 1: Import all required libraries
# ─────────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import os
import shutil
import gc
import json

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,           # NEW: needed to set dropout in model config
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

print("All libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")


All libraries imported successfully.
PyTorch version: 2.10.0+cu128
GPU available: True


In [3]:
print("Kaggle input root   : /kaggle/input")
print("Kaggle working root : /kaggle/working")

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_DATASET_FOLDER = None  # Set to a specific folder name if needed

# Automatically locate the split CSV files inside /kaggle/input
if PREFERRED_DATASET_FOLDER is not None:
    split_dir = KAGGLE_INPUT_ROOT / PREFERRED_DATASET_FOLDER
else:
    train_candidates = list(KAGGLE_INPUT_ROOT.rglob("train_split.csv"))
    val_candidates   = list(KAGGLE_INPUT_ROOT.rglob("val_split.csv"))
    test_candidates  = list(KAGGLE_INPUT_ROOT.rglob("test_split.csv"))

    if not train_candidates or not val_candidates or not test_candidates:
        raise FileNotFoundError(
            "Could not find train_split.csv, val_split.csv, and test_split.csv inside /kaggle/input. "
            "Attach your shared-splits dataset first."
        )

    split_dir = train_candidates[0].parent

TRAIN_PATH = split_dir / "train_split.csv"
VAL_PATH   = split_dir / "val_split.csv"
TEST_PATH  = split_dir / "test_split.csv"

print("Using split folder:", split_dir)
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH  :", VAL_PATH)
print("TEST_PATH :", TEST_PATH)

Kaggle input root   : /kaggle/input
Kaggle working root : /kaggle/working
Using split folder: /kaggle/input/datasets/prohorpaul/human-vs-gpt-5-4-paraphrased
TRAIN_PATH: /kaggle/input/datasets/prohorpaul/human-vs-gpt-5-4-paraphrased/train_split.csv
VAL_PATH  : /kaggle/input/datasets/prohorpaul/human-vs-gpt-5-4-paraphrased/val_split.csv
TEST_PATH : /kaggle/input/datasets/prohorpaul/human-vs-gpt-5-4-paraphrased/test_split.csv


In [4]:
# ─────────────────────────────────────────────
# CELL 3: Load datasets and inspect
# ─────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Reset index to avoid any index mismatches
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Dataset shapes:")
print("  Train shape      :", train_df.shape)
print("  Validation shape :", val_df.shape)
print("  Test shape       :", test_df.shape)


Dataset shapes:
  Train shape      : (10282, 4)
  Validation shape : (1470, 4)
  Test shape       : (2938, 4)


In [5]:
!pip install -q git+https://github.com/csebuetnlp/normalizer

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [6]:
from normalizer import normalize

for df in [train_df, val_df, test_df]:
    df["normalized_text"] = df["text"].astype(str).apply(normalize)

In [7]:
# ─────────────────────────────────────────────
# CELL 5: Keep only relevant columns
# ─────────────────────────────────────────────

train_df = train_df[["normalized_text", "label_encoded"]].copy()
val_df   = val_df[["normalized_text", "label_encoded"]].copy()
test_df  = test_df[["normalized_text", "label_encoded"]].copy()

for df in [train_df, val_df, test_df]:
    df.rename(columns={"normalized_text": "text", "label_encoded": "label"}, inplace=True)
    df["text"] = df["text"].fillna("").astype(str)

print("Columns kept: ['text', 'label']")
display(train_df.head())


Columns kept: ['text', 'label']


,text,label
0,রাজধানীর ক্যান্টনমেন্ট থানা এলাকায় গতকাল শুক্...,0
1,বাংলাদেশে নিযুক্ত মার্কিন রাষ্ট্রদূত ড্যান ডব্...,0
2,নীলফামারীর কিশোরগঞ্জে বিদ্যুতের তারে জড়িয়ে এ...,0
3,শেরপুরের ঝিনাইগাতী উপজেলায় গতকাল শুক্রবার সড়...,0
4,ফরিদপুরে সাপের কামড়ে দুই শিক্ষার্থী নিহত হয়ে...,0


In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

print("Datasets created:")
print("  Train:", train_dataset)
print("  Val  :", val_dataset)
print("  Test :", test_dataset)

Datasets created:
  Train: Dataset({
    features: ['text', 'label'],
    num_rows: 10282
})
  Val  : Dataset({
    features: ['text', 'label'],
    num_rows: 1470
})
  Test : Dataset({
    features: ['text', 'label'],
    num_rows: 2938
})
